In [0]:
%run ../utils/llm_helper

In [0]:


from pyspark.sql.functions import *
import json

# ================================
# STEP 1: READ QUARANTINE TABLES
# ================================
customers_df = spark.table("globalmart.silver.customers_quarantine")
orders_df = spark.table("globalmart.silver.orders_quarantine")
transactions_df = spark.table("globalmart.silver.transactions_quarantine")

# ================================
# STEP 2: GENERIC FUNCTION TO EXTRACT ISSUES
# ================================
def extract_dq_issues(df, entity_name):
    
    dq_cols = [
        f.name for f in df.schema.fields
        if f.name.startswith("_dq_") and str(f.dataType) == "BooleanType()"
    ]
    
    stack_expr = "stack({0}, {1}) as (dq_field, is_valid)".format(
        len(dq_cols),
        ", ".join([f"'{c}', {c}" for c in dq_cols])
    )
    
    exploded = df.selectExpr("*", stack_expr)
    
    failed = exploded.filter(col("is_valid") == False)
    
    result = (
        failed.groupBy("dq_field", "_dq_reason")
        .agg(count("*").alias("failed_count"))
        .withColumn("entity_name", lit(entity_name))
        .withColumnRenamed("dq_field", "field_name")   # rename HERE
        .withColumnRenamed("_dq_reason", "issue_type")
    )
    
    return result

# ================================
# STEP 3: COMBINE ALL ENTITIES
# ================================
df_list = []

if customers_df is not None:
    df_list.append(extract_dq_issues(customers_df, "customers"))

if orders_df is not None:
    df_list.append(extract_dq_issues(orders_df, "orders"))

if transactions_df is not None:
    df_list.append(extract_dq_issues(transactions_df, "transactions"))

final_df = df_list[0]
for d in df_list[1:]:
    final_df = final_df.unionByName(d)

# Normalize issue_type for consistency
final_df = final_df.withColumn("issue_type", trim(lower(col("issue_type"))))

display(final_df)

# ================================
# STEP 4: CALL LLM PER ISSUE (FIX)
# ================================
dq_data = final_df.toPandas().to_dict(orient="records")

results = []

for row in dq_data:
    prompt = f"""
You are a finance audit assistant.

Explain the business impact of this issue:

Entity: {row['entity_name']}
Field: {row['field_name']}
Issue: {row['issue_type']}
Failed Count: {row['failed_count']}

Return ONLY JSON:
{{
  "entity_name": "{row['entity_name']}",
  "issue_type": "{row['issue_type']}",
  "business_impact": "3-4 sentence explanation"
}}
"""

    llm_output = call_llm(prompt)

    try:
        parsed = json.loads(llm_output) if isinstance(llm_output, str) else llm_output
        
        # Ensure required keys
        parsed["entity_name"] = row["entity_name"]
        parsed["issue_type"] = row["issue_type"]
        
        if "business_impact" not in parsed:
            parsed["business_impact"] = "Impact could not be generated"
            
        results.append(parsed)
        
    except Exception as e:
        print("LLM failed for:", row, e)

# ================================
# STEP 5: CREATE AI DATAFRAME
# ================================
if results:
    ai_df = spark.createDataFrame(results)
else:
    ai_df = spark.createDataFrame([], schema="""
        entity_name STRING,
        issue_type STRING,
        business_impact STRING
    """)

# Normalize again before join
ai_df = ai_df.withColumn("issue_type", trim(lower(col("issue_type"))))



# ================================
# STEP 6: JOIN RESULTS
# ================================
final_output = final_df.join(
    ai_df,
    ["entity_name", "issue_type"],
    "left"
).withColumn(
    "generated_at",
    current_timestamp()
)



# ================================
# STEP 7: WRITE TO GOLD
# ================================
final_output.write.mode("overwrite").saveAsTable("globalmart.gold.dq_audit_report")

print("✅ DQ Audit Report Generated Successfully")

field_name,issue_type,failed_count,entity_name
_dq_email_ok,missing customer_id,385,customers
_dq_email_ok,missing customer_email,42,customers
_dq_segment_ok,missing customer_id,54,customers
_dq_id_ok,missing customer_id,539,customers


LLM failed for: {'field_name': '_dq_segment_ok', 'issue_type': 'missing customer_id', 'failed_count': 54, 'entity_name': 'customers'} Expecting value: line 1 column 1 (char 0)
✅ DQ Audit Report Generated Successfully


In [0]:
df=spark.read.table("globalmart.gold.dq_audit_report")
print("globalmart.gold.dq_audit_report count ",df.count())

df.display()


globalmart.gold.dq_audit_report count  7


entity_name,field_name,issue_type,failed_count,business_impact,generated_at
customers,_dq_email_ok,missing customer_id,385,"The absence of customer_id for 539 records compromises data integrity, making it difficult to accurately link customer profiles to transactions, contracts, and support tickets. This hampers reporting accuracy and can lead to misattributed revenue or expenses, potentially inflating or deflating financial statements. Additionally, the missing identifiers increase the risk of duplicate or erroneous entries, which can trigger audit findings and regulatory penalties. Overall, the issue threatens operational efficiency, financial reporting reliability, and compliance with data governance standards.",2026-03-26T10:45:58.498Z
customers,_dq_email_ok,missing customer_email,42,"The absence of email addresses for 42 customers hampers direct communication, limiting the effectiveness of marketing campaigns, order confirmations, and support notifications. This gap can lead to missed sales opportunities and delayed issue resolution, negatively affecting customer satisfaction and retention. Additionally, incomplete contact data may violate regulatory requirements for customer communication, exposing the company to compliance risks and potential penalties.",2026-03-26T10:45:58.498Z
customers,_dq_segment_ok,missing customer_id,54,"The absence of customer_id for 539 records compromises data integrity, making it difficult to accurately link customer profiles to transactions, contracts, and support tickets. This hampers reporting accuracy and can lead to misattributed revenue or expenses, potentially inflating or deflating financial statements. Additionally, the missing identifiers increase the risk of duplicate or erroneous entries, which can trigger audit findings and regulatory penalties. Overall, the issue threatens operational efficiency, financial reporting reliability, and compliance with data governance standards.",2026-03-26T10:45:58.498Z
customers,_dq_id_ok,missing customer_id,539,"The absence of customer_id for 539 records compromises data integrity, making it difficult to accurately link customer profiles to transactions, contracts, and support tickets. This hampers reporting accuracy and can lead to misattributed revenue or expenses, potentially inflating or deflating financial statements. Additionally, the missing identifiers increase the risk of duplicate or erroneous entries, which can trigger audit findings and regulatory penalties. Overall, the issue threatens operational efficiency, financial reporting reliability, and compliance with data governance standards.",2026-03-26T10:45:58.498Z
customers,_dq_email_ok,missing customer_id,385,"The absence of customer_id in 385 records compromises the integrity of the customer database, making it difficult to uniquely identify and track individual customers. This hampers accurate reporting, analytics, and customer relationship management, potentially leading to duplicate or incomplete customer profiles. Additionally, it can disrupt compliance with data governance standards and increase the risk of errors in billing, marketing, and support processes.",2026-03-26T10:45:58.498Z
customers,_dq_segment_ok,missing customer_id,54,"The absence of customer_id in 385 records compromises the integrity of the customer database, making it difficult to uniquely identify and track individual customers. This hampers accurate reporting, analytics, and customer relationship management, potentially leading to duplicate or incomplete customer profiles. Additionally, it can disrupt compliance with data governance standards and increase the risk of errors in billing, marketing, and support processes.",2026-03-26T10:45:58.498Z
customers,_dq_id_ok,missing customer_id,539,"The absence of customer_id in 385 records compromises the integrity of the customer database, making it difficult to uniquely identify and track individual customers. This hampers accurate reporting, analytics, and customer r